**Etapa 1:** Visão Geral e Proporção de Churn (Base de 80.000 Registros)

* **Objetivo**: Carregar a base de dados bancários, inspecionar o esquema e entender a proporção inicial de clientes ativos versus cancelados (Churn).

* **Ações realizadas**: 
  * Importação das bibliotecas essenciais e leitura das primeiras linhas
  * Descrição estatística das variáveis numéricas e categóricas.
  * Renomeação dos rótulos/colunas para facilitar o manuseio .
  * Contagem geral de clientes ativos e taxa de Churn


In [0]:
# Manipulação e estruturação de dados
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning e Métricas (caso vá avançar para modelagem preditiva)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [0]:
%sql
select * from workspace.default.bank_churn_dataset limit 30

In [0]:

%sql
describe workspace.default.bank_churn_dataset;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bank_churn_dataset_pt AS
SELECT 
    id                       AS id_cliente,
    full_name                AS nome_completo,
    credit_sco               AS score_credito,
    gender                   AS genero,
    age                      AS idade,
    occupation               AS profissao,
    balance                  AS saldo,
    monthly_ir               AS renda_mensal,
    address                  AS endereco,
    origin_province          AS provincia_origem,
    tenure_ye                AS anos_relacionamento,
    married                  AS casado,
    nums_card                AS num_cartoes,
    nums_service             AS num_servicos,
    CASE 
        WHEN active_member = TRUE THEN 'Ativo'
        ELSE 'Inativo'
    END                      AS membro_ativo,
    last_active_date         AS data_ultima_atividade,
    last_transaction_month   AS ultimo_mes_transacao,
    created_date             AS data_criacao,
    CASE WHEN exit = TRUE THEN 1 ELSE 0 END AS churn,
    customer_segment         AS segmento_cliente,
    engagement_score         AS score_engajamento,
    loyalty_level            AS nivel_fidelidade,
    digital_behavior         AS comportamento_digital,
    risk_score               AS score_risco,
    risk_segment             AS segmento_risco,
    cluster_group            AS grupo_cluster
FROM workspace.default.bank_churn_dataset;

Verificando se há dados nulos em nossa base de dados

In [0]:
df_pd = spark.read.table("workspace.default.bank_churn_dataset_pt").toPandas()
df_pd.info()

Iniciando perguntnas de negócio.


1 - Qual é a Taxa de Churn Geral da Base?

In [0]:
%sql
SELECT 
    churn,
    COUNT(*) AS total_clientes,
    ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()), 2) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY churn;

Databricks visualization. Run in Databricks to view.

Volume Total Analisado: 80.000 clientes cadastrados na base de dados.  

Clientes Cancelados (Churn): 14.400 clientes, o que representa 18% da base total.  


Clientes Ativos: 65.600 clientes, correspondendo a 82% do total

2 - Perfil Demográfico: Idade × Churn

In [0]:
%sql
SELECT 
    churn,
    COUNT(*) AS qtd_clientes,
    ROUND(AVG(idade), 1) AS media_idade,
    ROUND(MIN(idade), 0) AS idade_minima,
    ROUND(MAX(idade), 0) AS idade_maxima,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY churn;

Databricks visualization. Run in Databricks to view.

A diferença é de aproximadamente 2,2 anos a favor dos clientes ativos, indicando que o público que cancela o serviço tende a ser um pouco mais jovem do que a média da base geral.

3 - Saúde Financeira: Saldo e Renda Mensal × Churn

In [0]:
%sql
select
    churn,
    round(avg(saldo),2) as media_saldo,
    round(avg(renda_mensal),2) as media_renda_mensal,
    round(avg(score_credito),1) as media_score_credito,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()) AS percentual_total
from workspace.default.bank_churn_dataset_pt
group by churn

Databricks visualization. Run in Databricks to view.

Perfil do Churn vs. Ativos há uma diferença expressiva de capital: Os clientes que ficam na instituição tendem a ser muito mais rentáveis e possuem saldos substancialmente maiores em conta ( Mesmo sem o contexto exato da moeda ou o país de origem), a proporção é o que importa aqui.

Clientes que deram churn possuem um saldo médio e uma renda mensal drasticamente menores (cerca de 1/3 do valor) em comparação aos clientes que continuam ativos.)

  O Score de Crédito também reflete isso: quem vai embora possui uma pontuação de crédito inferior (média de 653 contra 691 dos ativos).  Conclusão de Negócio: O risco de churn está fortemente concentrado em clientes com menor poder aquisitivo (menor renda e saldo) e menor score de crédito dentro desta base.

In [0]:
%sql
SELECT 
    membro_ativo,
    churn,
    COUNT(*) AS total_clientes,
    round((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()),0) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY membro_ativo, churn
ORDER BY membro_ativo, churn;

Databricks visualization. Run in Databricks to view.

A esmagadora maioria dos cancelamentos vem do grupo de membros inativos (13.283 clientes, o equivalente a quase 92% de todo o churn da base).

O Poder dos Clientes Engajados: Ser um "membro ativo" é a melhor vacina contra o cancelamento. Quase ninguém que usa o banco com frequência decide ir embora (apenas 1,40% cancelaram).

Insighs:
Em vez de gastar energia tentando salvar todo mundo, o foco deve ser criar campanhas rápidas para acordar quem está com a conta parada antes que fechem a conta de vez.
